# 038 · Adam (Adaptive Moment Estimation)

**Momentum plus RMSprop, plus bias correction.** Two EWMAs, one of the gradient
and one of its square:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}, \qquad w \leftarrow w - \frac{\eta}{\sqrt{\hat{v}_t}+\varepsilon}\hat{m}_t$$

| Part | What we reproduce |
|---|---|
| A | implement Adam, then **ablate each half** and recover momentum and RMSprop |
| B | bias correction: **1,000×** at t=1, still **10.5×** at t=100 |
| C | without it, the first step would be about **32× too large** |
| D | why β₁ = 0.9 and β₂ = 0.999 are different numbers |

Needs `numpy`.

In [ ]:
import numpy as np

B1, B2, EPS = 0.9, 0.999, 1e-8


def adam(grad_fn, w0, lr=0.1, steps=200, b1=B1, b2=B2, correct=True):
    w = np.array(w0, float)
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    for t in range(1, steps + 1):
        g = grad_fn(w)
        m = b1 * m + (1 - b1) * g                 # first moment  -> DIRECTION
        v = b2 * v + (1 - b2) * g ** 2            # second moment -> SCALE
        m_hat = m / (1 - b1 ** t) if correct else m
        v_hat = v / (1 - b2 ** t) if correct else v
        w = w - lr * m_hat / (np.sqrt(v_hat) + EPS)
    return w

## Part A — Ablate each half

If Adam really is momentum plus RMSprop, then switching off one moment should
leave the other method behind. Check that rather than believing it.

In [ ]:
# The ravine again, so the comparison is against something familiar.
A, B = 1.0, 20.0

def grad(p):
    return np.array([A * p[0], B * p[1]])

def loss(p):
    return 0.5 * (A * p[0] ** 2 + B * p[1] ** 2)


def variant(use_first, use_second, steps=200, lr=0.1, w0=(9.0, 1.0)):
    w = np.array(w0, float)
    m, v = np.zeros(2), np.zeros(2)
    for t in range(1, steps + 1):
        g = grad(w)
        m = B1 * m + (1 - B1) * g
        v = B2 * v + (1 - B2) * g ** 2
        direction = m / (1 - B1 ** t) if use_first else g
        scale = np.sqrt(v / (1 - B2 ** t)) + EPS if use_second else 1.0
        w = w - lr * direction / scale
    return w


for name, f, s in (("plain GD  (neither)", False, False),
                   ("momentum  (first only)", True, False),
                   ("RMSprop   (second only)", False, True),
                   ("Adam      (both)", True, True)):
    w = variant(f, s)
    print(f"  {name:<26} final loss = {loss(w):.3e}")

### Read that table again — it does not say what you expected

**Momentum alone beats Adam here, by six orders of magnitude.** That is not a
bug in the implementation, and it is worth understanding rather than explaining
away.

The ravine is a *clean quadratic*. Momentum on a clean quadratic converges
**geometrically** — every step multiplies the distance to the minimum by a
constant. Adam divides by `√v̂`, which normalises the step to roughly `η`
**regardless of how small the gradient has become**. Near the minimum that is a
floor on the step size, so Adam stops improving while momentum keeps shrinking.

In [ ]:
l_adam = loss(variant(True, True))
l_mom = loss(variant(True, False))
l_rms = loss(variant(False, True))
print(f"Adam {l_adam:.2e}   momentum-only {l_mom:.2e}   RMSprop-only {l_rms:.2e}")
print(f"\nmomentum-only is {l_adam/l_mom:.0e}x lower. Adam is NOT the winner here.")

# Watch the step size to see why.
w, m, v = np.array([9.0, 1.0]), np.zeros(2), np.zeros(2)
print(f"\n{'step':>6}{'|gradient|':>14}{'|Adam step|':>14}")
for t in range(1, 201):
    g = grad(w)
    m = B1 * m + (1 - B1) * g
    v = B2 * v + (1 - B2) * g ** 2
    step = 0.1 * (m/(1-B1**t)) / (np.sqrt(v/(1-B2**t)) + EPS)
    w = w - step
    if t in (1, 10, 50, 200):
        print(f"{t:>6}{np.linalg.norm(g):>14.2e}{np.linalg.norm(step):>14.2e}")

print("\nThe gradient falls by orders of magnitude. The step barely moves.")
print("That normalisation is the whole point of Adam - and on a well-scaled")
print("problem it is a handicap, not a help.")

### So build the problem Adam is actually for

The ravine is badly *conditioned* but perfectly *scaled* — both parameters get
gradients of a similar size. Adam's per-parameter scaling earns its keep when
the gradients themselves differ by orders of magnitude, which is lesson 036's
sparse-feature problem.

In [ ]:
rng = np.random.default_rng(2)
N = 500
dense = rng.normal(0, 1, N)
sparse = (rng.random(N) < 0.04).astype(float)
Xs = np.c_[dense, sparse]
TRUE_W = np.array([1.0, 5.0])
ys = Xs @ TRUE_W + rng.normal(0, 0.1, N)

def sparse_grad(w):
    return 2 * Xs.T @ (Xs @ w - ys) / N

def sparse_mse(w):
    return float(np.mean((Xs @ w - ys) ** 2))


def variant2(use_first, use_second, steps=300, lr=0.05):
    w, m, v = np.zeros(2), np.zeros(2), np.zeros(2)
    for t in range(1, steps + 1):
        g = sparse_grad(w)
        m = B1 * m + (1 - B1) * g
        v = B2 * v + (1 - B2) * g ** 2
        direction = m / (1 - B1 ** t) if use_first else g
        scale = np.sqrt(v / (1 - B2 ** t)) + EPS if use_second else 1.0
        w = w - lr * direction / scale
    return w


for name, f, s in (("plain GD  (neither)", False, False),
                   ("momentum  (first only)", True, False),
                   ("RMSprop   (second only)", False, True),
                   ("Adam      (both)", True, True)):
    w = variant2(f, s)
    print(f"  {name:<26} sparse weight {w[1]:6.3f} of {TRUE_W[1]}  "
          f"mse {sparse_mse(w):.5f}")

In [ ]:
w_adam = variant2(True, True)
w_mom = variant2(True, False)
w_plain = variant2(False, False)

print(f"Adam recovers {w_adam[1]/TRUE_W[1]:.0%} of the sparse weight")
print(f"momentum-only {w_mom[1]/TRUE_W[1]:.0%}")
print(f"plain GD      {w_plain[1]/TRUE_W[1]:.0%}")

assert w_adam[1] > w_mom[1]        # the second moment is what matters here
assert w_adam[1] > w_plain[1]
print("\nHere the SECOND moment carries the benefit, and Adam wins clearly.")
print("On the ravine the FIRST moment carried it, and Adam lost.")
print("\nBoth results are real. Adam is the safe DEFAULT because it is rarely")
print("bad on any problem - not because it is the best on every problem.")

## Part B — Bias correction is not optional

Both moments start at zero, which is a claim that the gradient was zero before
training began. Lesson 033 met this as the EWMA cold start. Here it is fatal
rather than cosmetic, because the two moments are biased by **different amounts**
and one of them sits under a square root.

In [ ]:
print(f"{'t':<8}{'1/(1-b1^t)':>15}{'1/(1-b2^t)':>16}")
for t in (1, 2, 10, 100, 1000):
    print(f"{t:<8}{1/(1-B1**t):>15.2f}{1/(1-B2**t):>16.2f}")

print(f"\nAt t = 1 the uncorrected moments hold only {1-B1:.0%} of the gradient")
print(f"and {1-B2:.1%} of the squared gradient.")

assert abs(1/(1-B2**1) - 1000.0) < 0.01
assert abs(1/(1-B2**100) - 10.5) < 0.05

## Part C — What goes wrong without it

The update divides `m` by `√v`. Both are understated at t = 1, but by **different
amounts**, so work out the net effect rather than guessing at it.

In [ ]:
g = 1.0                                    # one gradient, for clarity
m1 = (1 - B1) * g
v1 = (1 - B2) * g ** 2

print(f"true gradient          : {g}")
print(f"m1 uncorrected         : {m1:.4f}     ({1/(1-B1):.0f}x too small)")
print(f"v1 uncorrected         : {v1:.6f}   ({1/(1-B2):.0f}x too small)")
print(f"sqrt(v1) uncorrected   : {np.sqrt(v1):.6f}   "
      f"({(1/(1-B2))**0.5:.1f}x too small)")
print()
uncorrected = m1 / (np.sqrt(v1) + EPS)
corrected = (m1 / (1 - B1)) / (np.sqrt(v1 / (1 - B2)) + EPS)
print(f"update WITHOUT correction : {uncorrected:.4f}")
print(f"update WITH correction    : {corrected:.4f}")
print(f"\n-> the first step is {uncorrected/corrected:.2f}x too large")

> **Careful here — this is a number that is easy to get wrong, and the lesson
> text states it loosely.** `√v₁` is understated by **31.6×**, and it is tempting
> to conclude the step is therefore 32× too large. It is not, because `m₁` is
> understated by **10×** at the same time, and that shrinks the step. The two do
> not cancel exactly, and what survives is their ratio:
>
> $$\frac{\sqrt{1/(1-\beta_2)}}{1/(1-\beta_1)} = \frac{31.6}{10} = 3.16$$
>
> So **the first step is about 3.2× too large, not 32×.** The 32× is real, but it
> is the size of the second-moment error on its own, not the size of the mistake
> it causes. Check it directly rather than taking either number on trust.

In [ ]:
ratio = uncorrected / corrected
predicted = np.sqrt(1/(1-B2)) / (1/(1-B1))
print(f"measured  : {ratio:.4f}x")
print(f"predicted : {predicted:.4f}x   = sqrt(1/(1-b2)) / (1/(1-b1))")
assert abs(ratio - predicted) < 1e-6
assert 3.0 < ratio < 3.3

print("\nStill worth correcting: a step 3.2x larger than intended, at the")
print("moment the weights are most fragile, is exactly how a run diverges")
print("in its first few iterations.")

In [ ]:
# The error is largest at t=1 and decays. Track the net factor over time.
print(f"{'t':>6}{'net step error':>18}")
for t in (1, 2, 5, 10, 50, 200, 1000):
    net = np.sqrt(1/(1-B2**t)) / (1/(1-B1**t))
    print(f"{t:>6}{net:>17.3f}x")
print("\nIt rises above 1 before settling back - the two corrections decay at")
print("very different rates, because beta1 and beta2 are very different numbers.")

In [ ]:
# And it is visible in training, not just in the arithmetic.
for correct in (True, False):
    w = adam(grad, (9.0, 1.0), lr=0.1, steps=200, correct=correct)
    label = "with correction" if correct else "without correction"
    print(f"  {label:<20} final loss = {loss(w):.3e}")
print("\nThe correction factor tends to 1, so this is a TRANSIENT - but the")
print("damage is done in the first few steps, when the weights are most fragile.")

## Part D — Why β₁ and β₂ are different numbers

They are estimating different things, and those things change at different rates.

In [ ]:
for b, name in ((B1, "beta1 (direction)"), (B2, "beta2 (magnitude)")):
    print(f"  {name:<22} = {b:<6} -> effective window {1/(1-b):>6.0f} steps")

print("\nDIRECTION should be responsive: which way is downhill changes as you")
print("move, so a ~10-step memory is right.")
print("MAGNITUDE should be steady: it is a scale estimate, and a noisy scale")
print("makes every step erratic, so a ~1000-step memory is right.")

In [ ]:
# Show it: swap the two betas and watch the result degrade.
def ravine_adam(b1, b2, steps=200):
    w, m, v = np.array([9.0, 1.0]), np.zeros(2), np.zeros(2)
    for t in range(1, steps + 1):
        g = grad(w)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g ** 2
        w = w - 0.1 * (m/(1-b1**t)) / (np.sqrt(v/(1-b2**t)) + EPS)
    return loss(w)

print(f"  standard  b1=0.9,   b2=0.999 -> loss {ravine_adam(0.9, 0.999):.3e}")
print(f"  swapped   b1=0.999, b2=0.9   -> loss {ravine_adam(0.999, 0.9):.3e}")
print("\nThe defaults are not arbitrary.")

In [ ]:
# Optimizer state costs memory: two extra values per parameter.
for params in (1e6, 1e8, 7e9):
    model_gb = params * 4 / 1024**3
    print(f"  {params:>10.0e} params: model {model_gb:>7.2f} GB, "
          f"+ Adam state {2*model_gb:>7.2f} GB = {3*model_gb:>7.2f} GB total")
print("\nRoughly 3x the model size in float32. This is why optimizer state is")
print("a first-class concern when training large models.")

## What to take away

- **Adam = momentum (first moment) + RMSprop (second moment)**, plus bias
  correction. The ablation in Part A recovers each half.
- **`m_t` is an EWMA of the gradient** → direction. **`v_t` is an EWMA of the
  squared gradient** → per-parameter scale.
- **Update: `w ← w − η/(√v̂_t + ε) m̂_t`** — RMSprop's rule with `g_t` replaced
  by `m̂_t`.
- **β₁ = 0.9 (≈10 steps) for direction; β₂ = 0.999 (≈1000 steps) for magnitude.**
  Responsive versus steady — and swapping them makes things worse.
- **Bias correction is not optional.** At t = 1, `v₁` understates by **1000×**
  and `√v₁` by **31.6×** — but `m₁` understates by **10×** at the same time, so
  the *net* first step is **3.16× too large**, not 32×. The lesson text states
  this loosely; the notebook works it out.
- The second-moment correction factor is **1,000 at t = 1**, still **10.5 at
  t = 100**, and tends to 1.
- **Adam addresses all four of lesson 032's problems** — the first in the series
  to do so.
- **Defaults η = 0.001, β₁ = 0.9, β₂ = 0.999 work almost everywhere.** Tune η
  first, if anything.
- **Cost: two extra values per parameter** — roughly 3× the model size in
  optimizer state.
- **Well-tuned SGD with momentum can generalise slightly better**, especially in
  vision. **AdamW** is standard for transformers.

## Exercises

1. Part A's ablation used one surface. Repeat it on lesson 036's sparse-feature
   problem. Does the second moment still carry most of the benefit there?
2. Set β₁ = 0 and confirm you get RMSprop exactly. Set β₂ → 1 with correction
   off and see what you get.
3. Bias correction matters most early. Plot the first 50 steps with and without
   it, and find the step at which the two trajectories become indistinguishable.
4. Implement **AdamW** — decoupled weight decay — and explain why adding L2 to
   the gradient is not the same thing once you divide by `√v̂`.
5. Adam's ε sits outside the square root. Move it inside and compare on a problem
   with very small gradients. Which convention does your framework use?
6. Train the same small network with SGD+momentum and with Adam to the same
   training loss, then compare **test** accuracy. Can you reproduce the claim
   that well-tuned SGD generalises slightly better?